# Демо проектных упражнений модуля 2 — Метрики сегментации и снимок специальных ID

**Демо-реализация, не готовое решение полномасштабного проекта.** Два примера на язык иллюстрируют агрегацию и опасность неверного знаменателя. `wordrun_proxy` считает серии букв/цифр и знаки, а не выполняет WordPiece; пробелы он отбрасывает. Китайская строка даёт одну серию букв, не одно лингвистическое слово. Это учебные сегментаторы, не рейтинг токенизаторов. Второе демо воспроизводит датированный снимок Harmony, не проверяя установленную библиотеку.
Все входы малы, детерминированы и работают без сети на CPU. Цель — проверить
интерфейсы, метрики, инварианты и структуру отчёта до расходования GPU/API-бюджета.

In [ ]:
from __future__ import annotations
import importlib.util, subprocess, sys
for package in ("numpy", "matplotlib"):
    if importlib.util.find_spec(package) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])


import math
import random
import time
from collections import Counter, defaultdict

import matplotlib.pyplot as plt
import numpy as np

SEED = 20260801
random.seed(SEED)
np.random.seed(SEED)
print(f"воспроизводимый тестовый запуск; seed={SEED}")
DEMO_LANGUAGE = 'ru'


## 1. Что именно стоит в знаменателе?

Сегментируем две короткие фразы на каждом из четырёх языков тремя способами.
Байтовый сегментатор выдаёт UTF-8-байты, символьный — кодовые точки, третий — серии
букв/цифр и отдельные знаки препинания. Последний отбрасывает пробелы, поэтому
его нельзя считать обратимым токенизатором. Сначала делим общее число единиц на
общее число серий букв/цифр, затем — на число выровненных предложений.

Перед запуском подумайте, почему китайская строка даёт одну серию и как это изменит
левую панель. Сравнивайте числа вместе с определением знаменателя: одинаковый цвет
на разных панелях не означает одинаковое значение. Две фразы показывают механику
метрики, но не позволяют судить об эффективности токенизации языка.

In [ ]:
INK = '#2E3440'
MUTED = '#4C566A'
BORDER = '#D8DEE9'
def plot_style():
    import matplotlib.pyplot as plt
    plt.rcParams.update({
        "font.family": "DejaVu Sans", "font.size": 12,
        "text.color": INK, "axes.labelcolor": INK, "axes.titlecolor": INK,
        "axes.titlesize": 17, "axes.titleweight": "bold", "axes.titlelocation": "left",
        "axes.labelsize": 13, "xtick.labelsize": 11, "ytick.labelsize": 11,
        "xtick.color": MUTED, "ytick.color": MUTED,
        "axes.edgecolor": BORDER, "axes.linewidth": .7,
        "axes.facecolor": "white", "figure.facecolor": "white", "savefig.facecolor": "white",
        "grid.color": BORDER, "grid.linewidth": .7, "axes.axisbelow": True,
        "legend.fontsize": 11, "legend.frameon": False, "savefig.dpi": 150,
        "xtick.major.size": 0, "ytick.major.size": 0,
    })

plot_style()

import re

corpus = {
    "en": ["models learn from tokens", "a short test"],
    "ru": ["модели учатся на токенах", "короткий тест"],
    "es": ["modelos aprenden de tokens", "una prueba corta"],
    "zh": ["模型从词元学习", "简短测试"],
}

def word_runs(text):
    return re.findall(r"[^\W_]+", text, flags=re.UNICODE)

tokenizers = {
    "byte": lambda s: list(s.encode("utf-8")),
    "char": list,
    "wordrun_proxy": lambda s: re.findall(r"[^\W_]+|[^\w\s]", s, flags=re.UNICODE),
}

# Одинаковый учебный смысл; корпус не является репрезентативным бенчмарком.
fertility = np.zeros((len(corpus), len(tokenizers)))
units_per_sentence = np.zeros_like(fertility)
denominators = {}
for i, (lang, lines) in enumerate(corpus.items()):
    n_runs = sum(len(word_runs(line)) for line in lines)
    denominators[lang] = n_runs
    n_bytes = sum(len(line.encode("utf-8")) for line in lines)
    print(lang, "предложений / серий букв и цифр / байтов:", len(lines), n_runs, n_bytes)
    for j, (name, encode) in enumerate(tokenizers.items()):
        n_units = sum(len(encode(line)) for line in lines)
        fertility[i, j] = n_units / n_runs
        units_per_sentence[i, j] = n_units / len(lines)
        print("  ", name, "единиц:", n_units, "единиц/предложение:", units_per_sentence[i, j])

tr = lambda ru, en: ru if DEMO_LANGUAGE == 'ru' else en
fig, axes = plt.subplots(1, 2, figsize=(11, 6), layout='constrained')
titles = [tr('Единиц на серию букв/цифр', 'Units per letter/digit run'),
          tr('Единиц на предложение', 'Units per aligned sentence')]
labels = [tr('Байты', 'Bytes'), tr('Кодовые\nточки', 'Code\npoints'),
          tr('Серии и\nзнаки', 'Runs and\npunctuation')]
for ax, values, title in zip(axes, [fertility, units_per_sentence], titles):
    im = ax.imshow(values, cmap='Blues', vmin=0, vmax=values.max(), aspect='auto')
    ax.set(xticks=range(len(tokenizers)), xticklabels=labels,
           yticks=range(len(corpus)), yticklabels=list(corpus))
    ax.grid(False)
    for i in range(values.shape[0]):
        for j in range(values.shape[1]):
            ax.text(j, i, f'{values[i, j]:.1f}', ha='center', va='center',
                    color='white' if values[i, j] > values.max() * .65 else INK)
    ax.set_title(title, fontsize=14, pad=14)
    fig.colorbar(im, ax=ax, location='bottom', shrink=.9,
                 label=tr('Единиц; своя шкала у каждой панели', 'Units; each panel has its own scale'))
fig.suptitle(tr('Выбор знаменателя меняет сравнение', 'The denominator changes the comparison'),
             fontsize=17, fontweight='bold', color=INK)
plt.show()
assert denominators["zh"] == 2  # Две целые строки, а не два лингвистических слова.
assert np.isfinite(fertility).all()
assert units_per_sentence.shape == (4, 3)
assert [len(word_runs(line)) for line in corpus["zh"]] == [1, 1]

## 2. Почему число имён не равно числу ID?

Ниже восстановим карту специальных имён из закреплённого исходного кода tiktoken
от 11 октября 2026 года. Два имени могут ссылаться на один ID: словарь Python
считает имена, а `set(special_to_id.values())` — уникальные значения. Перед запуском
сопоставьте диапазон резервных ID с `endofprompt`; после запуска объясните разницу
между 1091 именем и 1090 ID. Обычные токены занимают отдельный диапазон, поэтому
к числу уникальных специальных ID можно прибавить 199998.

In [ ]:
# Снимок: 2026-10-11. Источник: openai/tiktoken revision 4e71bbe0c078468e00fefbf94b39849389f346e5.
# Сверка установленной библиотеки включается отдельно в основном ноутбуке М2.
mergeable_ids = 199_998
protocol_to_id = {
    "<|startoftext|>": 199_998,
    "<|endoftext|>": 199_999,
    "<|return|>": 200_002,
    "<|constrain|>": 200_003,
    "<|channel|>": 200_005,
    "<|start|>": 200_006,
    "<|end|>": 200_007,
    "<|message|>": 200_008,
    "<|call|>": 200_012,
    "<|endofprompt|>": 200_018,
}
reserved_to_id = {
    "<|reserved_200000|>": 200_000,
    "<|reserved_200001|>": 200_001,
    "<|reserved_200004|>": 200_004,
    "<|reserved_200009|>": 200_009,
    "<|reserved_200010|>": 200_010,
    "<|reserved_200011|>": 200_011,
    **{f"<|reserved_{i}|>": i for i in range(200_013, 201_088)},
}
special_to_id = protocol_to_id | reserved_to_id
special_ids = set(special_to_id.values())
aliases = {}
for name, token_id in special_to_id.items():
    aliases.setdefault(token_id, []).append(name)
aliases = {token_id: names for token_id, names in aliases.items() if len(names) > 1}

print("ID обычных токенов:", mergeable_ids)
print("специальные имена / уникальные ID:", len(special_to_id), len(special_ids))
print("имена одного ID:", aliases)
print("всего уникальных ID:", mergeable_ids + len(special_ids))
assert len(protocol_to_id) == 10
assert len(reserved_to_id) == 1_081
assert len(special_to_id) == 1_091
assert len(special_ids) == 1_090
assert aliases == {200_018: ["<|endofprompt|>", "<|reserved_200018|>"]}
assert mergeable_ids + len(special_ids) == 201_088

## Переход к полному проекту

Замените учебный корпус на одинаковые строки FLORES для десяти языков, а упрощённые токенизаторы — на закреплённые версии реальных. Сохраняйте число токенов на выровненное предложение, число байтов и ревизию каждого артефакта. Для fertility задайте отдельные правила лингвистической сегментации языков; не выдавайте regex-серии за китайские слова.

### Минимальный отчёт

1. Версии данных, модели, кода и seed.
2. Основная метрика, а также качество, стоимость, задержка и память там, где это применимо.
3. Базовая линия, одинаковый бюджет и хотя бы один отрицательный контроль.
4. Неудачные прогоны и границы обоснованных выводов.